In [9]:
import duckdb
import pandas as pd

fact_orders = pd.read_csv('/content/Fact_Orders.csv')
dim_customers = pd.read_csv('/content/Dim_Customers.csv')
dim_products = pd.read_csv('/content/Dim_Products.csv')

con = duckdb.connect(':memory:')
con.register('fact_orders', fact_orders)
con.register('dim_customers', dim_customers)
con.register('dim_products', dim_products)

/tmp/ipykernel_833/2179966304.py:4: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  fact_orders = pd.read_csv('/content/Fact_Orders.csv')


In [10]:
query1="""
SELECT * FROM fact_orders """
duckdb.query(query1).df()

,item_id,increment_id,customer_id,sku,created_at,qty_ordered,net_revenue,status,payment_method,sales_event
0,211133.0,100147444,2,kcc_Buy 2 Frey Air Freshener & Get 1 Kasual Bo...,2016-07-01,1,240.0,canceled,cod,Normal
1,211134.0,100147445,3,Ego_UP0017-999-MR0,2016-07-01,1,2450.0,canceled,cod,Normal
2,211135.0,100147446,4,kcc_krone deal,2016-07-01,1,60.0,complete,cod,Normal
3,211136.0,100147447,5,BK7010400AG,2016-07-01,2,1110.0,refunded,cod,Normal
4,211137.0,100147448,6,UK_Namkino All In One 200 Gms,2016-07-01,1,80.0,canceled,cod,Normal
...,...,...,...,...,...,...,...,...,...,...
476588,905198.0,100562381,111132,MEFPAK5B360B03C6B72,2018-08-28,1,999.0,complete,customercredit,Normal
476589,905199.0,100562382,8123,MATINF59BAB39FDBEF1,2018-08-28,2,13520.0,pending,voucher,Normal
476590,905200.0,100562383,115325,WOFVAL59D5EA84167F9-M,2018-08-28,1,400.0,complete,cod,Normal
476591,905202.0,100562384,115325,WOFNIG5B4D7EB0E9FDD-L,2018-08-28,1,499.0,complete,cod,Normal


In [11]:
query2 = """
SELECT
    status,
    SUM(net_revenue) AS total_revenue
FROM fact_orders
GROUP BY status
ORDER BY total_revenue DESC;
"""
duckdb.query(query2).df()

,status,total_revenue
0,complete,3.337390e+08
1,canceled,3.321787e+08
2,refunded,8.231492e+07
3,payment_review,1.120387e+05
4,pending,8.218600e+04
5,processing,2.896770e+04
6,holded,2.501950e+04
7,None,2.242900e+04
8,pending_paypal,7.565500e+03
9,\n,1.620000e+02


In [12]:
query3 = """
SELECT
    payment_method,
    COUNT(item_id) AS total_orders,
    SUM(CASE WHEN status = 'canceled' THEN 1 ELSE 0 END) AS canceled_orders,
    ROUND(
        100.0 * SUM(CASE WHEN status = 'canceled' THEN 1 ELSE 0 END) / NULLIF(COUNT(item_id), 0),
        2
    ) AS cancellation_rate_pct
FROM fact_orders
GROUP BY payment_method
ORDER BY cancellation_rate_pct DESC;
"""
duckdb.query(query3).df()

,payment_method,total_orders,canceled_orders,cancellation_rate_pct
0,mygateway,504,488.0,96.83
1,ublcreditcard,532,359.0,67.48
2,easypay_ma,11117,7212.0,64.87
3,bank_transfer,88008,53509.0,60.80
4,mobile_wallet,93554,52878.0,56.52
5,voucher,22524,11483.0,50.98
6,productcredit,121,10.0,8.26
7,cod,252603,16375.0,6.48
8,cashatdoorstep,711,5.0,0.70
9,customercredit,6867,45.0,0.66


In [13]:
query_4 = """
SELECT
    fo.sales_event,
    COUNT(fo.item_id) AS total_items_sold,
    SUM(fo.net_revenue) AS total_revenue
FROM fact_orders fo
JOIN dim_products dp ON fo.sku = dp.sku
WHERE fo.sales_event IN ('White Friday', 'Normal')
GROUP BY fo.sales_event
ORDER BY total_revenue DESC;
"""
duckdb.query(query_4).df()

,sales_event,total_items_sold,total_revenue
0,Normal,428772,6.299313e+08
1,White Friday,29287,4.493556e+07


In [14]:
query_5 = """
SELECT
    dp.category_name_1 AS category,
    SUM(fo.net_revenue) AS total_revenue
FROM fact_orders fo
JOIN dim_products dp ON fo.sku = dp.sku
WHERE fo.sales_event = 'Ramadan'
GROUP BY dp.category_name_1
ORDER BY total_revenue DESC
LIMIT 5;
"""
duckdb.query(query_5).df()

,category,total_revenue
0,Mobiles & Tablets,5.644401e+07
1,Women's Fashion,3.659382e+06
2,Men's Fashion,3.516576e+06
3,Appliances,2.239089e+06
4,Beauty & Grooming,2.065474e+06


In [15]:
query_6 = """
WITH DailyMetrics AS (
    SELECT
        CAST(created_at AS DATE) AS sales_date,
        COUNT(DISTINCT increment_id) AS total_orders,
        COUNT(DISTINCT customer_id) AS unique_buyers,
        SUM(net_revenue) AS daily_revenue
    FROM fact_orders
    WHERE status = 'complete'
    GROUP BY CAST(created_at AS DATE)
)
SELECT
    sales_date,
    total_orders,
    unique_buyers,
    ROUND(daily_revenue, 2) AS daily_revenue,
    ROUND(daily_revenue / NULLIF(total_orders, 0), 2) AS calculated_aov
FROM DailyMetrics
ORDER BY sales_date DESC
LIMIT 10;
"""
duckdb.query(query_6).df()

,sales_date,total_orders,unique_buyers,daily_revenue,calculated_aov
0,2018-08-28,22,21,38975.75,1771.63
1,2018-08-27,88,84,180947.92,2056.23
2,2018-08-26,57,57,131268.40,2302.95
3,2018-08-25,75,70,141796.80,1890.62
4,2018-08-24,69,59,197617.10,2864.02
5,2018-08-23,42,40,77732.10,1850.76
6,2018-08-22,21,21,33215.00,1581.67
7,2018-08-21,49,49,99830.60,2037.36
8,2018-08-20,60,55,190172.40,3169.54
9,2018-08-19,73,65,157944.50,2163.62


In [20]:
query_7 = """
WITH MonthlyRevenue AS (
    SELECT
        DATE_TRUNC('month', CAST(created_at AS TIMESTAMP)) AS sales_month,
        SUM(net_revenue) AS current_month_revenue
    FROM fact_orders
    WHERE status = 'complete'
    GROUP BY DATE_TRUNC('month', CAST(created_at AS TIMESTAMP))
)
SELECT
    sales_month,
    ROUND(current_month_revenue, 2) AS current_revenue,
    ROUND(
        LAG(current_month_revenue, 1) OVER (ORDER BY sales_month ASC),
        2
    ) AS prior_month_revenue,
    ROUND(
        current_month_revenue - LAG(current_month_revenue, 1) OVER (ORDER BY sales_month ASC),
        2
    ) AS mom_absolute_growth,
    ROUND(
        100.0 * (current_month_revenue - LAG(current_month_revenue, 1) OVER (ORDER BY sales_month ASC))
        / NULLIF(LAG(current_month_revenue, 1) OVER (ORDER BY sales_month ASC), 0),
        2
    ) AS mom_growth_pct
FROM MonthlyRevenue
ORDER BY sales_month ASC;
"""
duckdb.query(query_7).df()

,sales_month,current_revenue,prior_month_revenue,mom_absolute_growth,mom_growth_pct
0,2016-07-01,4464306.09,NaN,NaN,NaN
1,2016-08-01,4971587.87,4464306.09,507281.78,11.36
2,2016-09-01,7064903.01,4971587.87,2093315.14,42.11
3,2016-10-01,5312343.97,7064903.01,-1752559.04,-24.81
4,2016-11-01,30043620.66,5312343.97,24731276.69,465.54
5,2016-12-01,6649924.38,30043620.66,-23393696.28,-77.87
6,2017-01-01,8867606.17,6649924.38,2217681.79,33.35
7,2017-02-01,10925819.88,8867606.17,2058213.71,23.21
8,2017-03-01,16734528.82,10925819.88,5808708.94,53.16
9,2017-04-01,10902359.75,16734528.82,-5832169.07,-34.85


In [21]:
query_8 = """
WITH CategoryRevenue AS (
    SELECT
        dp.category_name_1 AS category,
        SUM(fo.net_revenue) AS total_revenue,
        RANK() OVER (ORDER BY SUM(fo.net_revenue) DESC) AS revenue_rank
    FROM fact_orders fo
    JOIN dim_products dp ON fo.sku = dp.sku
    WHERE fo.status = 'complete'
    GROUP BY dp.category_name_1
)
SELECT
    category,
    ROUND(total_revenue, 2) AS total_revenue,
    revenue_rank
FROM CategoryRevenue
WHERE revenue_rank <= 3
ORDER BY revenue_rank ASC;
"""
duckdb.query(query_8).df()

,category,total_revenue,revenue_rank
0,Mobiles & Tablets,70161727.47,1
1,Appliances,51879976.72,2
2,Women's Fashion,49104961.03,3


In [22]:
query_9 = """
SELECT DISTINCT
    customer_id,
    MIN(created_at) OVER (PARTITION BY customer_id) AS first_order_date
FROM fact_orders
ORDER BY customer_id ASC
LIMIT 10;
"""
duckdb.query(query_9).df()

,customer_id,first_order_date
0,2,2016-07-01
1,3,2016-07-01
2,4,2016-07-01
3,5,2016-07-01
4,6,2016-07-01
5,7,2016-07-01
6,9,2016-07-01
7,10,2016-07-01
8,11,2016-07-01
9,12,2016-07-01
